# TubeWatch Channel + Playlist End-to-End Tester

按 cell 顺序操作：先运行到频道选择界面，输入频道、加载并选择一个来源；然后运行下方端到端测试 cell 查看 3 个视频的处理状态和字幕 sample；确认结果后再运行最后的清理 cell，精确删除本次数据库记录和测试产物。

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import uuid

import ipywidgets as widgets
from IPython.display import display

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("无法定位 TubeWatch 项目根目录。")

project_root = find_project_root(Path.cwd().resolve())
state_db = project_root / "data" / "tubewatch.sqlite3"
tester_output_root = (project_root / "output" / "tester").resolve()
command_environment = dict(os.environ)
command_environment["PYTHONIOENCODING"] = "utf-8"

def run_cli(arguments):
    return subprocess.run(
        [sys.executable, "-m", "tubewatch", *arguments],
        cwd=project_root, capture_output=True, text=True, encoding="utf-8",
        env=command_environment,
    )

def failure_detail(result):
    return (result.stderr.strip() or result.stdout.strip()).replace("\r\n", "\n")

## 配置

`run_tubescribe` 默认保持 `False`；运行真实端到端测试 cell 前，还必须在界面中显式勾选确认。

In [ ]:
limit = 3
sample_lines = 20
run_tubescribe = True  # 保持默认关闭；界面中还需显式确认

## 可选：无效输入验证（不访问网络）

In [ ]:
invalid_cases = [
    ([""], "source_url 不能为空"),
    (["@bad/handle"], "有效的 YouTube 频道 handle"),
    (["https://www.youtube.com/playlist"], "list 参数"),
    (["check", "https://example.com/not-a-channel"], "YouTube 频道 URL"),
    (["playlists", "https://example.com/not-a-channel"], "YouTube 频道 URL"),
    (["process", "--limit", "0"], "limit 必须是正整数"),
    (["process", "--source-url", "@example"], "必须同时提供"),
]
for arguments, expected_text in invalid_cases:
    result = run_cli(arguments)
    detail = failure_detail(result)
    assert result.returncode == 2, detail
    assert expected_text in detail and "Traceback" not in detail, detail
print("无效输入提示验证通过。")

In [ ]:
channel_input = widgets.Text(
    description="频道", placeholder="@handle 或完整频道 URL",
    layout=widgets.Layout(width="720px"),
)
load_button = widgets.Button(description="加载播放列表", button_style="primary")
source_dropdown = widgets.Dropdown(
    description="来源", options=[("请先加载频道", "")],
    value="", disabled=True, layout=widgets.Layout(width="720px"),
)
playlist_output = widgets.Output()
loaded_channel = None
tubescribe_confirmation = widgets.Checkbox(
    value=run_tubescribe,
    description="确认下一测试 cell 真实调用 TubeScribe",
    indent=False,
)

def reset_source_selection():
    global loaded_channel
    loaded_channel = None
    source_dropdown.options = [("请先加载频道", "")]
    source_dropdown.value = ""
    source_dropdown.disabled = True

def on_channel_change(change):
    if change["name"] == "value":
        reset_source_selection()
        playlist_output.clear_output()

def load_channel_playlists(_):
    global loaded_channel
    channel = channel_input.value.strip()
    reset_source_selection()
    load_button.disabled = True
    try:
        with playlist_output:
            playlist_output.clear_output()
            if not channel:
                print("请先输入频道 handle 或完整频道 URL。")
                return
            print("正在加载频道公开播放列表……")
            result = run_cli(["playlists", channel, "--json"])
            playlist_output.clear_output()
            if result.returncode != 0:
                print(failure_detail(result))
                return
            playlists = json.loads(result.stdout)
            options = [("请选择来源", ""), ("频道视频（不选择播放列表）", "__channel__")]
            options.extend(
                (f"{item['title']}（{item['playlist_id']}）", item["url"])
                for item in playlists
            )
            source_dropdown.options = options
            source_dropdown.value = ""
            source_dropdown.disabled = False
            loaded_channel = channel
            print(f"已加载 {len(playlists)} 个公开播放列表，请明确选择一个来源。")
    finally:
        load_button.disabled = False

channel_input.observe(on_channel_change, names="value")
load_button.on_click(load_channel_playlists)
display(widgets.VBox([
    channel_input, load_button, source_dropdown, tubescribe_confirmation, playlist_output,
]))

## 运行真实端到端测试

在上方完成频道加载、来源选择和真实处理确认后，运行下一 cell。它会读取并处理 3 个全新视频，逐项显示状态，并输出成功字幕的前 20 个非空行。测试数据会保留到最后的清理 cell。

In [ ]:
if globals().get("active_test_run") is not None:
    raise RuntimeError("上一次测试尚未清理，请先运行最后的清理 cell。")

channel = channel_input.value.strip()
selected_value = source_dropdown.value
if loaded_channel != channel or not selected_value:
    raise RuntimeError("请先在上方加载频道并明确选择频道视频或一个播放列表。")
if not tubescribe_confirmation.value:
    raise RuntimeError("请先勾选真实 TubeScribe 处理确认。")

selected_source = channel if selected_value == "__channel__" else selected_value
tester_output_root.mkdir(parents=True, exist_ok=True)
run_directory = (tester_output_root / uuid.uuid4().hex).resolve()
if run_directory.parent != tester_output_root:
    raise RuntimeError("测试输出目录越过了 output/tester 安全边界。")
raw_directory = run_directory / "raw"
cleaned_directory = run_directory / "cleaned"
raw_directory.mkdir(parents=True)
cleaned_directory.mkdir(parents=True)
active_test_run = {
    "source_url": None, "video_ids": [], "run_directory": run_directory,
}

check_result = run_cli([
    "check", selected_source, "--limit", str(limit),
    "--state-db", str(state_db), "--json",
])
if check_result.returncode != 0:
    raise RuntimeError(failure_detail(check_result))
checked = json.loads(check_result.stdout)
checked_source_url = checked["source_url"]
test_video_ids = [item["video_id"] for item in checked["new_videos"]]
active_test_run["source_url"] = checked_source_url
active_test_run["video_ids"] = test_video_ids
if checked["fetched_count"] != limit or len(test_video_ids) != limit:
    raise RuntimeError(
        f"端到端测试需要 {limit} 个全新视频；本次读取 "
        f"{checked['fetched_count']} 个、新增 {len(test_video_ids)} 个。请运行清理 cell。"
    )

process_arguments = [
    "process", "--limit", str(limit), "--state-db", str(state_db),
    "--raw-dir", str(raw_directory), "--cleaned-dir", str(cleaned_directory),
    "--source-url", checked_source_url, "--json",
]
for video_id in test_video_ids:
    process_arguments.extend(["--video-id", video_id])
process_result = run_cli(process_arguments)
if not process_result.stdout.strip():
    raise RuntimeError(f"{failure_detail(process_result)}\n请运行清理 cell。")
processed = json.loads(process_result.stdout)
if processed["attempted_count"] != limit or processed["pending_remaining"] != 0:
    raise RuntimeError("TubeScribe 未完整尝试本次指定的三个视频；请运行清理 cell。")
processed_video_ids = [item["video_id"] for item in processed["results"]]
if set(processed_video_ids) != set(test_video_ids) or len(processed_video_ids) != limit:
    raise RuntimeError("TubeScribe 返回的视频集合与本次精确过滤集合不一致；请运行清理 cell。")

failed_messages = []
for item in processed["results"]:
    print(f"\n[{item['status']}] {item['title']}")
    if item["status"] == "succeeded":
        raw_path = Path(item["raw_path"]).resolve()
        cleaned_path = Path(item["cleaned_path"]).resolve()
        if not raw_path.is_relative_to(raw_directory) or not cleaned_path.is_relative_to(cleaned_directory):
            raise RuntimeError("TubeScribe 返回了测试目录之外的产物路径；请运行清理 cell。")
        if not raw_path.is_file() or not cleaned_path.is_file():
            raise RuntimeError("TubeScribe 报告成功，但字幕产物不存在；请运行清理 cell。")
        lines = [
            line for line in cleaned_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
        if not lines:
            raise RuntimeError("TubeScribe 清理文本为空；请运行清理 cell。")
        print(f"字幕语言：{item['language_code']}")
        print("字幕 sample：")
        print("\n".join(lines[:sample_lines]))
    elif item["status"] == "no_subtitles":
        print(f"无可下载字幕：{item['error_message']}")
    else:
        failed_messages.append(item["error_message"] or item["video_id"])
if failed_messages or process_result.returncode != 0:
    raise RuntimeError("TubeScribe 普通失败：" + "；".join(failed_messages) + "。请运行清理 cell。")
print("\n测试结果和字幕 sample 已保留。确认后请运行下方清理 cell。")

## 清理本次测试

查看完测试状态和字幕 sample 后运行下一 cell。它只删除 `active_test_run` 记录的精确视频集合和本次唯一的字幕目录；测试 cell 报错时也应运行这里。

In [ ]:
test_run = globals().get("active_test_run")
if test_run is None:
    raise RuntimeError("当前没有待清理的测试；请先运行端到端测试 cell。")

cleanup_errors = []
checked_source_url = test_run["source_url"]
test_video_ids = test_run["video_ids"]
run_directory = Path(test_run["run_directory"]).resolve()
if checked_source_url and test_video_ids:
    cleanup_arguments = [
        "cleanup-test", checked_source_url, "--state-db", str(state_db), "--json",
    ]
    for video_id in test_video_ids:
        cleanup_arguments.extend(["--video-id", video_id])
    cleanup_result = run_cli(cleanup_arguments)
    if cleanup_result.returncode != 0:
        cleanup_errors.append(failure_detail(cleanup_result))
    else:
        cleanup = json.loads(cleanup_result.stdout)
        if cleanup["removed_count"] != len(test_video_ids):
            cleanup_errors.append(
                f"数据库只清理了 {cleanup['removed_count']}/{len(test_video_ids)} 条记录。"
            )
        else:
            print(f"数据库已精确清理 {cleanup['removed_count']} 条本次测试记录。")
if run_directory.exists():
    if run_directory.parent != tester_output_root:
        cleanup_errors.append("拒绝删除 output/tester 边界之外的目录。")
    else:
        shutil.rmtree(run_directory)
        print("本次测试字幕目录已删除。")
if cleanup_errors:
    raise RuntimeError("；".join(cleanup_errors))
active_test_run = None
print("本次测试已清理完成。")